In [ ]:
# LANGGRAPH + AZURE OPENAI — MULTI-AGENT MEDICAL ROUTING

from typing import TypedDict
from langgraph.graph import StateGraph, START, END
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import AzureChatOpenAI

# Azure OpenAI
llm = AzureChatOpenAI(
    azure_endpoint="https://YOUR-RESOURCE.openai.azure.com/",
    api_key="YOUR_API_KEY", api_version="2024-10-21",
    azure_deployment="gpt-4.1", temperature=0
)

# Tools
def web_search(query: str) -> str:
    return f"Web Search Result: {query}"

def pubmed_search(query: str) -> str:
    return f"PubMed Result: {query}"

# Shared State
class AgentState(TypedDict):
    question: str
    route: str
    answer: str

# Supervisor Agent
supervisor_prompt = ChatPromptTemplate.from_messages([
    ("system", """You are a supervisor agent.
Route medical/health/disease/symptoms/drugs/treatment/
diagnosis/clinical questions to medical.
Route everything else to general.
Reply ONLY: medical OR general."""),
    ("human", "{question}")
])

def supervisor(state: AgentState) -> dict:
    response = (supervisor_prompt | llm).invoke(
        {"question": state["question"]}
    )
    route = response.content.strip().lower()
    route = route if route in {"medical", "general"} else "general"
    print(f"Supervisor Route: {route}")
    return {"route": route}

# Medical Agent
def medical_agent(state: AgentState) -> dict:
    result = pubmed_search(state["question"])
    return {"answer": f"Medical Agent: {result}"}

# General Agent
def general_agent(state: AgentState) -> dict:
    result = web_search(state["question"])
    return {"answer": f"General Agent: {result}"}

# Conditional Routing
def route_agent(state: AgentState):
    return state["route"]

# Build Multi-Agent Graph
builder = StateGraph(AgentState)

builder.add_node("supervisor", supervisor)
builder.add_node("medical_agent", medical_agent)
builder.add_node("general_agent", general_agent)

builder.add_edge(START, "supervisor")

builder.add_conditional_edges(
    "supervisor",
    route_agent,
    {
        "medical": "medical_agent",
        "general": "general_agent"
    }
)

builder.add_edge("medical_agent", END)
builder.add_edge("general_agent", END)

graph = builder.compile()

# Invoke
result = graph.invoke({
    "question": "What is the treatment for diabetes?",
    "route": "",
    "answer": ""
})

print(f"\nFinal Answer: {result['answer']}")